In [ ]:
# 3. MODELING — Lasso ➜ OLS backward (p > 0.5) — 5 cibles
# ---------------------------------------------------------
# Entrées  : data/df_dummies_2019.csv (features + 5 cibles)
# Sorties  :
#   - models/<target>_lasso.pkl               (sklearn Pipeline)
#   - Geodechet/model_paths/<target>.pkl      (statsmodels OLS final pour l'app)
#   - docs/model_metrics.csv                  (R2, RMSE, MAE, R2_CV)
#   - docs/OLS_<target>.txt / _pvalues.csv    (rapports OLS finaux)
#   - models/features_SCHEMA.csv              (ordre des features Lasso)
# ---------------------------------------------------------

from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import statsmodels.api as sm

# --- Dossiers ---
ROOT = Path("..").resolve()
DATA = ROOT / "data"
MODELS = ROOT / "models"
DOCS = ROOT / "docs"
APP_MD = ROOT / "Geodechet" / "model_paths"  # OLS pour l'app
for d in (MODELS, DOCS, APP_MD):
    d.mkdir(parents=True, exist_ok=True)

# --- Dataset ---
DFP = DATA / "df_dummies_2019.csv"
sample = DFP.read_text(encoding="utf-8", errors="ignore")[:400]
sep = ";" if sample.count(";") > sample.count(",") else ","
df = pd.read_csv(DFP, sep=sep).drop(columns=["Unnamed: 0"], errors="ignore")
print("Loaded:", DFP.name, "| shape:", df.shape)
display(df.head(2))

In [ ]:
# --- Cibles (exactement 5) ---
TARGETS = [
    "Déchets_verts",
    "Matériaux_recyclables",
    "Encombrants",
    "Total_autres_dechets",
    "Déblais_gravats",
]
for t in TARGETS:
    assert t in df.columns, f"Cible manquante : {t}"

# --- Features : tout sauf cibles (on garde les ID si utiles à l'app ? Ici on les exclut côté modèle) ---
maybe_ids = [
    c for c in ["année", "annee", "Région", "region", "Code_Dpt"] if c in df.columns
]
feature_cols = [c for c in df.columns if c not in TARGETS + maybe_ids]
assert len(feature_cols) > 0, "Aucune feature détectée."
print("Nb features :", len(feature_cols))

In [ ]:
# --- Helpers (compat sklearn anciennes versions) ---
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def make_lasso_pipeline():
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("lasso", LassoCV(cv=5, random_state=42, max_iter=20000)),
        ]
    )


def evaluate(model, X_train, y_train, X_test, y_test):
    y_pred = model.predict(X_test)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring="r2")
    return {
        "R2_test": float(r2_score(y_test, y_pred)),
        "RMSE_test": rmse(y_test, y_pred),
        "MAE_test": float(mean_absolute_error(y_test, y_pred)),
        "R2_CV_mean": float(np.mean(cv_scores)),
        "R2_CV_std": float(np.std(cv_scores)),
    }

In [ ]:
# --- Backward elimination OLS (seuil p-value = 0.5, comme votre pratique) ---
def backward_ols(X_df: pd.DataFrame, y: pd.Series, p_threshold: float = 0.5):
    """
    Applique une élimination arrière : on fit un OLS,
    on enlève la feature avec p-value max si > p_threshold,
    on répète jusqu'à ce que toutes les p-values <= p_threshold.
    Retourne: (model_ols_final, kept_features_list)
    """
    kept = list(X_df.columns)
    while True:
        Xc = sm.add_constant(X_df[kept], has_constant="add")
        model = sm.OLS(y, Xc, missing="drop").fit()
        pvals = model.pvalues.drop(labels=["const"], errors="ignore")
        if pvals.empty:
            break
        worst = pvals.idxmax()
        worst_p = pvals.loc[worst]
        if np.isnan(worst_p) or worst_p <= p_threshold:
            # stop si plus de p>seuil
            break
        # retire la variable la moins significative
        kept.remove(worst)
    # refit final
    Xc = sm.add_constant(X_df[kept], has_constant="add")
    final_model = sm.OLS(y, Xc, missing="drop").fit()
    return final_model, kept

In [ ]:
# --- Entrainement : Lasso ➜ OLS backward ---
RANDOM_STATE = 42
TEST_SIZE = 0.2
P_THRESHOLD = 0.5  # seuil de p-value pour l'élimination manuelle

registry = []
metrics_rows = []

for target in TARGETS:
    print("=" * 80)
    print("Cible :", target)

    y = pd.to_numeric(df[target], errors="coerce")
    X = df[feature_cols].copy()

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    # 1) LassoCV (sélection auto)
    lasso_pipe = make_lasso_pipeline()
    lasso_pipe.fit(X_train, y_train)

    # Features non nulles côté Lasso (importance structurelle)
    lasso = lasso_pipe.named_steps["lasso"]
    non_zero = lasso.coef_ != 0
    feats_lasso = [f for f, keep in zip(X.columns, non_zero) if keep]
    if len(feats_lasso) == 0:
        # si le Lasso est ultra strict -> garde un top-N par variance comme garde-fou
        feats_lasso = list(X.var().sort_values(ascending=False).head(30).index)

    # Éval Lasso
    m = evaluate(lasso_pipe, X_train, y_train, X_test, y_test)
    m["target"] = target
    metrics_rows.append(m)
    print("Metrics Lasso :", m)

    # Sauvegarde Lasso
    lasso_path = MODELS / f"{target}_lasso.pkl"
    joblib.dump({"model": lasso_pipe, "features": feature_cols}, lasso_path)

    # 2) OLS backward p>0.5 sur le SOUS-ENSEMBLE issu du Lasso (comme votre process)
    ols_final, kept_feats = backward_ols(X[feats_lasso], y, p_threshold=P_THRESHOLD)

    # Exports OLS pour l'app
    ols_path = APP_MD / f"{target}.pkl"
    with open(ols_path, "wb") as f:
        pickle.dump(ols_final, f)

    # Rapports OLS
    (DOCS / f"OLS_{target}.txt").write_text(
        ols_final.summary().as_text(), encoding="utf-8"
    )
    pvals_df = (
        ols_final.pvalues.drop(labels=["const"], errors="ignore")
        .sort_values()
        .to_frame("p_value")
        .reset_index()
        .rename(columns={"index": "feature"})
    )
    pvals_df.to_csv(DOCS / f"OLS_{target}_pvalues.csv", index=False, encoding="utf-8")

    # Registry
    registry.append(
        {
            "target": target,
            "lasso_model_path": str(lasso_path.relative_to(ROOT)),
            "ols_model_path": str(ols_path.relative_to(ROOT)),
            "n_features_lasso": int(len(feats_lasso)),
            "n_features_ols": int(len(kept_feats)),
            "metrics_lasso": m,
            "p_threshold": P_THRESHOLD,
        }
    )

print("\n✅ Entraînement terminé pour les 5 cibles (Lasso + OLS backward).")

In [ ]:
# --- Exports globaux : métriques (Lasso), registry, schema features ---
pd.DataFrame(metrics_rows).sort_values("target").to_csv(
    DOCS / "model_metrics.csv", index=False, encoding="utf-8"
)

with open(MODELS / "registry.json", "w", encoding="utf-8") as f:
    json.dump(registry, f, ensure_ascii=False, indent=2)

# Schéma "features Lasso" (ordre de X complet, utile si tu réutilises le Lasso ailleurs)
pd.Series(feature_cols, name="feature").to_csv(
    MODELS / "features_SCHEMA.csv", index=False, encoding="utf-8"
)

print("Saved ->", DOCS / "model_metrics.csv")
print("Saved ->", MODELS / "registry.json")
print("Saved ->", MODELS / "features_SCHEMA.csv")
print("Saved OLS (app) ->", APP_MD)

In [ ]:
# --- Aperçu des résultats (Lasso) ---
pd.read_csv(DOCS / "model_metrics.csv")